# Benchmark, Inference, and Placeholder Unlearning

Load baseline checkpoints, preview predictions, run the `retain_finetune` placeholder unlearning method, and benchmark all model families.

In [ ]:
from pathlib import Path

import json
import matplotlib.pyplot as plt

from data_utils import resolve_pipeline_paths

from notebook_workflows import (
    preview_checkpoint_predictions,
    results_to_dataframe,
    run_benchmark_notebook_workflow,
    run_retain_finetune_placeholder,
)


In [ ]:
# Centralized config
DATASET = "mufac"  # or "cifar10"
DEVICE = "auto"
NUM_BANK_SEEDS = 3
CHECKPOINT_DIR = Path("checkpoints")
BASELINE_TRAIN_FAMILY = "baseline_train"
BASELINE_RETRAIN_FAMILY = "baseline_retrain"
PLACEHOLDER_FAMILY = "placeholder_unlearn"
PLACEHOLDER_EPOCHS = 1
PLACEHOLDER_LR = None  # defaults to the selected baseline_train metadata
PLACEHOLDER_WEIGHT_DECAY = None  # defaults to the selected baseline_train metadata
PLACEHOLDER_BATCH_SIZE = None  # defaults to the selected baseline_train metadata
EVAL_BATCH_SIZE = 128 if DATASET == "cifar10" else 64
IMAGE_SIZE = 32 if DATASET == "cifar10" else 224
EFFICIENCY_RATIO = 0.2
PLACEHOLDER_USE_WANDB = False
BENCHMARK_USE_WANDB = True
BENCHMARK_WANDB_PROJECT = "benchmark"

DATA_ROOT = None
TASK_MANIFEST = None
SAMPLES_CSV = None


In [ ]:
_, resolved_task_manifest, _ = resolve_pipeline_paths(DATASET, SAMPLES_CSV, TASK_MANIFEST, DATA_ROOT)
TASK_ID = json.loads(resolved_task_manifest.read_text(encoding="utf-8"))["task_id"]
baseline_train_dir = CHECKPOINT_DIR / DATASET / TASK_ID / BASELINE_TRAIN_FAMILY
baseline_train_metadata = json.loads((baseline_train_dir / "seed_0.json").read_text(encoding="utf-8"))

placeholder_lr = (
    PLACEHOLDER_LR if PLACEHOLDER_LR is not None else baseline_train_metadata["learning_rate"]
)
placeholder_weight_decay = (
    PLACEHOLDER_WEIGHT_DECAY
    if PLACEHOLDER_WEIGHT_DECAY is not None
    else baseline_train_metadata["weight_decay"]
)
placeholder_batch_size = (
    PLACEHOLDER_BATCH_SIZE
    if PLACEHOLDER_BATCH_SIZE is not None
    else baseline_train_metadata["batch_size"]
)

placeholder_outputs = run_retain_finetune_placeholder(
    dataset=DATASET,
    base_family_dir=baseline_train_dir,
    output_family_name=PLACEHOLDER_FAMILY,
    num_bank_seeds=NUM_BANK_SEEDS,
    epochs=PLACEHOLDER_EPOCHS,
    lr=placeholder_lr,
    weight_decay=placeholder_weight_decay,
    batch_size=placeholder_batch_size,
    image_size=IMAGE_SIZE,
    checkpoint_dir=CHECKPOINT_DIR,
    data_root=DATA_ROOT,
    task_manifest=TASK_MANIFEST,
    samples_csv=SAMPLES_CSV,
    device_name=DEVICE,
    use_wandb=PLACEHOLDER_USE_WANDB,
)

placeholder_outputs


In [ ]:
prediction_preview = preview_checkpoint_predictions(
    dataset=DATASET,
    checkpoint_path=placeholder_outputs["seed_bank"][0]["checkpoint_path"],
    loader_name="test",
    sample_count=5,
    batch_size=EVAL_BATCH_SIZE,
    image_size=IMAGE_SIZE,
    data_root=DATA_ROOT,
    task_manifest=TASK_MANIFEST,
    samples_csv=SAMPLES_CSV,
    device_name=DEVICE,
)

display(results_to_dataframe(prediction_preview))


In [ ]:
benchmark_outputs = run_benchmark_notebook_workflow(
    dataset=DATASET,
    checkpoint_dir=CHECKPOINT_DIR,
    baseline_train_family=BASELINE_TRAIN_FAMILY,
    baseline_retrain_family=BASELINE_RETRAIN_FAMILY,
    placeholder_family=PLACEHOLDER_FAMILY,
    efficiency_ratio=EFFICIENCY_RATIO,
    data_root=DATA_ROOT,
    task_manifest=TASK_MANIFEST,
    samples_csv=SAMPLES_CSV,
    device_name=DEVICE,
    batch_size=EVAL_BATCH_SIZE,
    image_size=IMAGE_SIZE,
    use_wandb=BENCHMARK_USE_WANDB,
    wandb_project=BENCHMARK_WANDB_PROJECT,
)

benchmark_outputs


In [ ]:
family_df = results_to_dataframe(
    [
        {"family": family_name, **summary}
        for family_name, summary in benchmark_outputs["family_summaries"].items()
    ]
)
display(family_df)

comparison_df = results_to_dataframe(
    [
        {
            "family": family_name,
            "forgetting_quality": comparison["forgetting_quality"],
            "passed_efficiency_cutoff": comparison["passed_efficiency_cutoff"],
            "final_score": comparison["final_score"],
        }
        for family_name, comparison in benchmark_outputs["comparisons_to_reference"].items()
    ]
)
display(comparison_df)


In [ ]:
if hasattr(family_df, "plot"):
    ax = family_df.plot(x="family", y=["retain_accuracy_mean", "test_accuracy_mean"], kind="bar", title=f"{DATASET} family utility")
    ax.set_ylabel("Accuracy")
    plt.show()

if hasattr(comparison_df, "plot") and len(comparison_df) > 0:
    ax = comparison_df.plot(x="family", y="forgetting_quality", kind="bar", title=f"{DATASET} forgetting quality")
    ax.set_ylabel("Forgetting quality")
    plt.show()
